This code detects the location of starting point of an office;

Input: List of offices + Corresponding offices 

=> Output: List of offices with coordinate information

In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json

In [61]:
def Detect_Office(Json,Office):

    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText'][0]) or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office2(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if ( d['inferText'][0] == '〇') or( d['inferText'][0] == '○') or ( d['inferText'][0] == '0')or ( d['inferText'][0] == 'O') or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(file_data):
    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

def Clova2(file_data):
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\Temp.jpg"
    with open(temp_path, "wb") as path:
        path.write(file_data)
    
    stream = open(temp_path, "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

    HH,WW=img.shape[:2]
    cropped=img[0:200,0:WW]
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    
    with open(temp_path+"Temp.jpg",'rb') as f:
        image = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(image).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

In [363]:
def hconcat_resize_min(im_list, interpolation=cv2.INTER_CUBIC):
    h_min = min(im.shape[0] for im in im_list)
    im_list_resize = [cv2.resize(im, (int(im.shape[1] * h_min / im.shape[0]), h_min), interpolation=interpolation)
                      for im in im_list]
    return cv2.hconcat(im_list_resize)

def Check(df):
    for n in range(1,len(df)):
        try:
            
            #Extract key info of office
            Row  = df.iloc[n]

            ###Collect Location information###
            Dept=Row["Dept"]
            Office=Row["Office"]

            print('['+str(n)+',"'+Dept+'","'+Office+'"]')
            StrPage=dta[Year][Dept][Office]['StrPage']
            StrPart=dta[Year][Dept][Office]['StrPart']
            StrLoc=int(dta[Year][Dept][Office]['StrLocation'])

            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+StrPart+".jpg",'rb')
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            oldimg=img.copy()[0:200,0:WW]

            EndLoc=int(dta[Year][Dept][Office]['EndLocation'])
            EndPage=int(dta[Year][Dept][Office]['EndPage'])
            EndPart=dta[Year][Dept][Office]['EndPart']


            #Start#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(StrPage)+"\\"+StrPart+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(StrLoc,0),(StrLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            #End#
            path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
            stream = open(path+"Page"+"{:03d}".format(EndPage)+"\\"+EndPart+".jpg",'rb')            
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

            Annotate=img.copy()[0:200,0:WW]
            cv2.line(Annotate,(EndLoc,0),(EndLoc,HH),(255,0,0),2)
            Annotate=cv2.copyMakeBorder(Annotate,20,20,20,20,cv2.BORDER_CONSTANT)
            oldimg=hconcat_resize_min((Annotate,oldimg))

            cv2.imshow(Dept+Office,oldimg)
            cv2.waitKey(0)
        except:
            print('Failed at '+Dept+Office)
    cv2.destroyAllWindows()
    cv2.waitKey(0)

In [96]:
def Show(n,Part,StrLoc):
    Row=df.iloc[n]
    Page=Row['Page']
    
    path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+Part+".jpg",'rb')
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    
    HH,WW=img.shape[:2]
    cv2.line(img,(StrLoc,0),(StrLoc,HH),(225,0,0),2)
    
    cv2.imshow('Projection',img)
    cv2.waitKey(0)

#Step 1

Fill in years to refer.
Remember to keep it as a string. NOT as float.

In [539]:
Year='1941'
Showa='16'
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
os.chdir(path)
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

In [499]:
file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

#Step 2

The following code will extract the location of offices.

In [65]:
n=0
FailedList=[]
#Extract key info of office
Row  = df.iloc[n]

Page=int(Row["Page"])
Office=Row["Office"]
Dept=Row['Dept']

###Insert Starting page information to motherframe###
dta[Year][Dept][Office]={}
dta[Year][Dept][Office]["Starting_Page"]=Page
print(dta[Year][Dept][Office])

###Collect Location information###
#Part1
with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
    file_data = f.read()
#Convert to json via CLOVA
Json=Clova(file_data)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["StrLocation"]='NA'
else:
    dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
    dta[str(Year)][Dept][Office]["StrPart"]='Top'
    print(Office+ 'success: at row'+str(n))

#Part2
with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
    file_data = f.read()
#Convert to json via CLOVA
Json=Clova(file_data)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
if XCoord_Unit==None:
    #Add to motherframe
    dta[str(Year)][Dept][Office]["StrLocation"]='NA'
    print(Office+ 'failed')
    FailedList.append(list((n,Dept,Office)))
else:
    dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
    dta[str(Year)][Dept][Office]["StrPart"]='Btm'
    print(Office+ 'success: at row'+str(n))

{'Starting_Page': 3}
秘書課success: at row0
秘書課success: at row0


In [66]:
#Test code| Version 2#
#Show Working office#
for n in range(1,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']
    print('['+str(n)+',"'+Dept+'","'+Office+'"]')
    
    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Insert Starting page information to motherframe###
    dta[Year][Dept][Office]={}
    dta[Year][Dept][Office]["StrPage"]=Page
    print(dta[Year][Dept][Office])

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova(file_data)
    except:
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

[1,"総務局","文書課"]
{'StrPage': 4}
Top Loc356
文書課success: at row1
[2,"総務局","人事課"]
{'StrPage': 5}
Top Loc375
人事課success: at row2
[3,"総務局","吏務課"]
{'StrPage': 5}
Top Loc146
吏務課success: at row3
[4,"総務局","議案課"]
{'StrPage': 6}
Top Loc154
議案課success: at row4
[5,"総務局","企画課"]
{'StrPage': 6}
Top LocNone
Bottom LocNone
企画課failed
[6,"総務局","都市計画課"]
{'StrPage': 7}
Top LocNone
Bottom LocNone
都市計画課failed
[7,"総務局","統計課"]
{'StrPage': 8}
Top LocNone
Bottom Loc371
統計課success: at row7
[8,"総務局","情報課"]
{'StrPage': 10}
Top Loc241
情報課success: at row8
[9,"監査部","市務監察課"]
{'StrPage': 10}
Top Loc157
市務監察課success: at row9
[10,"監査部","区務監察課"]
{'StrPage': 11}
Top LocNone
Bottom LocNone
区務監察課failed
[11,"財務局","会計課"]
{'StrPage': 12}
Top Loc360
会計課success: at row11
[12,"財務局","主計課"]
{'StrPage': 13}
Top Loc474
主計課success: at row12
[13,"財務局","主税課"]
{'StrPage': 13}
Top Loc474
主税課success: at row13
[14,"財務局","公債課"]
{'StrPage': 15}
Top LocNone
Bottom Loc252
公債課success: at row14
[15,"財務局","用品課"]
{'StrPage': 16}
Top Loc339
用品課success: 

#Step 3

Check for errors.
Manually type in the index, department name, and office name of erroneous department-office pair to a seperate list.

In [67]:
FailedList2=[]
for Office in FailedList:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    Json=Clova2(file_data)
    
    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office2(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova2(file_data)
    except:
        print(Office+ 'failed')
        FailedList2.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office2(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList2.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

Top LocNone
Bottom Loc288
企画課success: at row5
Top Loc319
都市計画課success: at row6
Top Loc82
区務監察課success: at row10
Top Loc151
学校営業課success: at row20
Top Loc358
町会課success: at row24
Top Loc223
学校職員課success: at row35
Top LocNone
Bottom LocNone
視学課failed
Top Loc476
社会教育課success: at row39
Top Loc207
療養所success: at row52
Top Loc151
監理課success: at row53
Top Loc471
会計課success: at row58
Top Loc471
分院及学校success: at row59
Top LocNone
Bottom Loc244
副収入役室success: at row69
Top LocNone
Bottom LocNone
道路建設課failed
Top Loc303
治水工事課success: at row75
Top LocNone
Bottom Loc391
計書課success: at row80
Top Loc286
臨時飛行場建設事務所success: at row84
Top LocNone
Bottom LocNone
計書課failed
Top LocNone
Bottom LocNone
気局failed
Top LocNone
Bottom Loc84
車両課success: at row104
Top Loc41
第一調整課success: at row106
Top Loc184
第二調整課success: at row107
Top LocNone
Bottom LocNone
営業課failed
Top Loc358
臨時電源調査課success: at row113


In [68]:
for Office in FailedList2:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    dta[str(Year)][Dept][Office]["StrLocation"]=0
    dta[str(Year)][Dept][Office]["StrPart"]='Top'
    dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
    dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
    dta[str(Year)][ExDept][ExOffice]["EndLocation"]=0
    dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       

In [83]:
Check(df)

[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"総務局","情報課"]
[9,"監査部","市務監察課"]
[10,"監査部","区務監察課"]
[11,"財務局","会計課"]
[12,"財務局","主計課"]
[13,"財務局","主税課"]
[14,"財務局","公債課"]
[15,"財務局","用品課"]
[16,"財務局","購買課"]
[17,"財務局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"建築部","装置課"]
[22,"市民局","庶務課"]
[23,"市民局","区政課"]
[24,"市民局","町会課"]
[25,"市民局","體力課"]
[26,"市民局","公園課"]
[27,"総動員部","第一課"]
[28,"総動員部","第二課"]
[29,"防衛局","庶務課"]
[30,"防衛局","計書課"]
[31,"防衛局","防衛課"]
[32,"防衛局","施設課"]
[33,"防衛局","防火改修課"]
[34,"教育局","庶務課"]
[35,"教育局","学校職員課"]
[36,"教育局","学務課"]
[37,"教育局","視学課"]
[38,"教育局","青年教育課"]
[39,"教育局","社会教育課"]
[40,"教育局","学校体育課"]
[41,"教育局","教育研究所"]
[42,"厚生局","庶務課"]
[43,"厚生局","軍事援護課"]
[44,"厚生局","児童課"]
[45,"厚生局","保護課"]
[46,"厚生局","福利課"]
[47,"厚生局","衛生課"]
[48,"厚生局","防疫課"]
[49,"清掃部","管理課"]
[50,"清掃部","作業課"]
[51,"清掃部","計書課"]
[52,"清掃部","療養所"]
[53,"清掃部","監理課"]
[54,"清掃部","醫務課"]
[55,"養育院","庶務課"]
[56,"養育院","監護課"]
[57,"養育院","医務課"]
[58,"養育院","会計課"]

KeyError: 'EndLocation'

Type in the department-offices with errors.

In [88]:
MissAlign=[[29,"防衛局","庶務課"],
          [30,"防衛局","計書課"],
          [33,"防衛局","防火改修課"],
          [47,"厚生局","衛生課"],
          [52,"清掃部","療養所"],
          [53,"清掃部","監理課"],
          [54,"清掃部","醫務課"],
          [59,"養育院","分院及学校"],
          [64,"消費経済部","物価課"],
           [80,"港湾局","計書課"],
           [88,"水道局","給水課"],
           [95,"水道局","水源林事務所"],
           [96,"水道局","気局"],
           [97,"電気局","医務局"],
           [102,"運輸部","運輸課"],
           [106,"交通調整部","第一調整課"],
           [110,"電燈部","営業課"]
          ]

Type1=[[0,"Admin","秘書課"],
       [3,"総務局","吏務課"],
      [9,"監査部","市務監察課"],
       [28,"総動員部","第一課"],
       [28,"総動員部","第二課"],
       [29,"防衛局","庶務課"],
       [31,"防衛局","防衛課"],
       [38,"教育局","青年教育課"],
       [40,"教育局","学校体育課"],
       [41,"教育局","教育研究所"],
       [43,"厚生局","軍事援護課"],
       [45,"厚生局","保護課"],
       [51,"清掃部","計書課"],
       [65,"消費経済部","配給課"],
       [69,"中央卸売市場","副収入役室"],
       [72,"土木局","道路建設課"],
       [74,"土木局","河川課"],
       [76,"土木局","土木試験所"],
       [77,"土木局","土木出張所"],
       [83,"港湾局","副収入役室"],
       [84,"港湾局","臨時飛行場建設事務所"],
       [89,"水道局","計書課"],
       [91,"水道局","小河内貯水池建設事務所"],
       [93,"水道局","庶務課"],
       [103,"運輸部","乗客課"],
       [107,"交通調整部","第二調整課"],
       [108,"交通調整部","技術課"],
       [112,"電燈部","電力課"]
      ]+FailedList2+MissAlign

#Step 4

See how much errors were observed in the first trial.

In [129]:
Type1_Fail=[]
for Office in Type1:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Collect Location information###
    #Part1
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Top.jpg",'rb') as f:
        file_data = f.read()
    #Convert to json via CLOVA
    Json=Clova2(file_data)
    
    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office2(Json,Office)
    print('Top Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Top'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))
    
    if XCoord_Unit!=None:
        continue
        
    #Part2
    ##Read image for first page##
    with open("Page"+"{:03d}".format(Page)+"\\"+"Btm.jpg",'rb') as f:
        file_data = f.read()

    #Convert to json via CLOVA
    try:
        Json=Clova2(file_data)
    except:
        print(Office+ 'failed')
        Type1_Fail.append(list((n,Dept,Office)))
        continue

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office2(Json,Office)
    print('Bottom Loc'+str(XCoord_Unit))
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["StrLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]='NA'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        Type1_Fail.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["StrLocation"]=XCoord_Unit
        dta[str(Year)][Dept][Office]["StrPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndPage"]=Page
        dta[str(Year)][ExDept][ExOffice]["EndPart"]='Btm'
        dta[str(Year)][ExDept][ExOffice]["EndLocation"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row'+str(n))

Top Loc119
秘書課success: at row0
Top Loc146
吏務課success: at row3
Top Loc240
市務監察課success: at row9
Top LocNone
Bottom Loc235
第二課success: at row28
Top LocNone
Bottom Loc235
第二課success: at row28
Top Loc349
庶務課success: at row29
Top Loc258
防衛課success: at row31
Top Loc416
青年教育課success: at row38
Top Loc442
学校体育課success: at row40
Top LocNone
Bottom Loc325
教育研究所success: at row41
Top LocNone
Bottom LocNone
軍事援護課failed
Top LocNone
Bottom Loc148
保護課success: at row45
Top LocNone
Bottom Loc471
計書課success: at row51
Top Loc201
配給課success: at row65
Top LocNone
Bottom Loc244
副収入役室success: at row69
Top LocNone
Bottom LocNone
道路建設課failed
Top LocNone
Bottom LocNone
河川課failed
Top Loc392
土木試験所success: at row76
Top Loc392
土木出張所success: at row77
Top Loc286
副収入役室success: at row83
Top Loc286
臨時飛行場建設事務所success: at row84
Top LocNone
Bottom LocNone
計書課failed
Top LocNone
Bottom Loc178
小河内貯水池建設事務所success: at row91
Top Loc465
庶務課success: at row93
Top Loc202
乗客課success: at row103
Top Loc184
第二調整課success: at row107
Top Loc

In [126]:
FailRate=len(Type1)/len(df)
if FailRate<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-FailRate))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
display(DF)
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv',index=False)

残念...Success Rate is 0.5652173913043479


,Year,Parts,WritePage,Page,PageFin,Office,OfficeFin,Unit,UnitFin,Position,PositionFin,Data
0,1934,1,1,0.985714286,.,0.9,.,NaN,.,0.08,.,.
1,1935,1,1,1,.,0.984375,.,NaN,.,-0.09375,.,.
2,1936,1,0.235294118,.,.,0.235294118,.,NaN,.,.,.,.
3,1937,1,1,1,.,0.865671642,.,NaN,.,.,.,.
4,1938,1,1,1,.,1,.,0.988095238,.,0.98816568,.,0.619957537
5,1939,1,1,1,.,0.927835052,.,1,.,.,.,.
6,1940,2,1,.,.,0.991803279,.,NaN,.,.,.,.
7,1941,2,1,1,.,0.565217,.,NaN,.,.,.,.
8,1942,2,1,1,.,.,.,NaN,.,.,.,.
9,1943,2,1,.,.,.,.,.,.,.,.,.


In [128]:
k=0
n,Office,Dept=Type1_Fail[k][0],Type1_Fail[k][2],Type1_2[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,'Btm',320)
dta[Year][Dept][Office]['StrLocation']=320
Remaining=list(Type1_Fail.remove(Type1_2[0]))
len(Remaining)

土木局 道路建設課


TypeError: 'NoneType' object is not iterable

In [136]:
k=1
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,'Btm',345)
dta[Year][Dept][Office]['StrLocation']=345
dta[Year][ExDept][ExOffice]['EndLocation']=345
dta[Year][Dept][Office]['StrPart']='Btm'
dta[Year][ExDept][ExOffice]['EndPart']='Btm'


総務局 吏務課


In [144]:
k=2
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,'Btm',120)
cv2.destroyAllWindows()
cv2.waitKey(0)
dta[Year][Dept][Office]['StrLocation']=120
dta[Year][ExDept][ExOffice]['EndLocation']=120
dta[Year][Dept][Office]['StrPart']='Btm'
dta[Year][ExDept][ExOffice]['EndPart']='Btm'


監査部 市務監察課


In [148]:
k=3
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,'Btm',500)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=500
dta[Year][ExDept][ExOffice]['EndLocation']=500
dta[Year][Dept][Office]['StrPart']='Btm'
dta[Year][ExDept][ExOffice]['EndPart']='Btm'

総動員部 第一課


In [151]:
k=4
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,'Btm',240)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=240
dta[Year][ExDept][ExOffice]['EndLocation']=240
dta[Year][Dept][Office]['StrPart']='Btm'
dta[Year][ExDept][ExOffice]['EndPart']='Btm'

総動員部 第二課


In [156]:
k=5
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Show(n,'Btm',115)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=115
dta[Year][ExDept][ExOffice]['EndLocation']=115
dta[Year][Dept][Office]['StrPart']='Btm'
dta[Year][ExDept][ExOffice]['EndPart']='Btm'

防衛局 庶務課


In [163]:
k=6
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=260
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

防衛局 防衛課


In [164]:
k=7
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=260
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 青年教育課


In [169]:
k=8
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=345
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 学校体育課


In [172]:
k=9
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=365
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 教育研究所


In [177]:
k=10
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=320
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

厚生局 軍事援護課


In [181]:
k=11
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=150
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

厚生局 保護課


In [185]:
k=12
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=470
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

清掃部 計書課


38

In [190]:
k=13
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=540
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

消費経済部 配給課


37

In [193]:
k=14
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=500
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

中央卸売市場 副収入役室


36

In [197]:
k=15
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=50
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 道路建設課


35

In [201]:
k=16
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=100
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 河川課


34

In [205]:
k=17
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=440
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 土木試験所


33

In [209]:
k=18
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=180
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 土木出張所


32

In [213]:
k=19
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=290
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 副収入役室


31

In [214]:
k=20
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=180
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 臨時飛行場建設事務所


30

In [218]:
k=21
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=75
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

水道局 計書課


29

In [223]:
k=22
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=180
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

水道局 小河内貯水池建設事務所


28

In [227]:
k=23
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=480
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

水道局 庶務課


27

In [232]:
k=24
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=210
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

運輸部 乗客課


26

In [234]:
k=25
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=500
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

交通調整部 第二調整課


25

In [235]:
k=26
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=190
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

交通調整部 技術課


24

In [238]:
k=27
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=360
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電燈部 電力課


23

In [240]:
k=28
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=380
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

教育局 視学課


22

In [242]:
k=29
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=50
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

土木局 道路建設課


21

In [244]:
k=30
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=70
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

水道局 計書課


20

In [254]:
k=31
n,Office,Dept=Type1[k][0],'総務課','電気局'
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=375
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year]['電気局']['総務課']['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気局 総務課


19

In [258]:
k=32
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=480
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電燈部 営業課


18

In [260]:
k=33
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=110
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

防衛局 庶務課


17

In [264]:
k=34
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=155
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

防衛局 計書課


16

In [269]:
k=35
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=225
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

防衛局 防火改修課


15

In [270]:
k=36
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=275
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

厚生局 衛生課


14

In [288]:
k=37
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=160
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

清掃部 療養所


13

In [294]:
k=38
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=100
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

清掃部 監理課


12

In [299]:
k=39
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=170
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

清掃部 醫務課


11

In [305]:
k=40
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=145
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

養育院 分院及学校


10

In [307]:
k=41
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=165
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

消費経済部 物価課


9

In [312]:
k=42
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=330
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

港湾局 計書課


8

In [316]:
k=43
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=240
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

水道局 給水課


7

In [339]:
k=46
n,Office,Dept=Type1[k][0],'総務課',Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=375
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電気局 総務課


4

In [344]:
k=47
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=280
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

運輸部 運輸課


3

In [349]:
k=48
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=440
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

交通調整部 第一調整課


2

In [354]:
k=49
n,Office,Dept=Type1[k][0],Type1[k][2],Type1[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=470
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
len(Type1)-k

電燈部 営業課


1

In [365]:
Check(df)

[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"総務局","情報課"]
[9,"監査部","市務監察課"]
[10,"監査部","区務監察課"]
[11,"財務局","会計課"]
[12,"財務局","主計課"]
[13,"財務局","主税課"]
[14,"財務局","公債課"]
[15,"財務局","用品課"]
[16,"財務局","購買課"]
[17,"財務局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"建築部","装置課"]
[22,"市民局","庶務課"]
[23,"市民局","区政課"]
[24,"市民局","町会課"]
[25,"市民局","體力課"]
[26,"市民局","公園課"]
[27,"総動員部","第一課"]
[28,"総動員部","第二課"]
[29,"防衛局","庶務課"]
[30,"防衛局","計書課"]
[31,"防衛局","防衛課"]
[32,"防衛局","施設課"]
[33,"防衛局","防火改修課"]
[34,"教育局","庶務課"]
[35,"教育局","学校職員課"]
[36,"教育局","学務課"]
Failed at 教育局学務課
[37,"教育局","視学課"]
[38,"教育局","青年教育課"]
[39,"教育局","社会教育課"]
[40,"教育局","学校体育課"]
[41,"教育局","教育研究所"]
[42,"厚生局","庶務課"]
Failed at 厚生局庶務課
[43,"厚生局","軍事援護課"]
[44,"厚生局","児童課"]
[45,"厚生局","保護課"]
[46,"厚生局","福利課"]
[47,"厚生局","衛生課"]
[48,"厚生局","防疫課"]
[49,"清掃部","管理課"]
[50,"清掃部","作業課"]
[51,"清掃部","計書課"]
[52,"清掃部","療養所"]
[53,"清掃部","監理課"]
[54,"清掃部","醫務課"]
[55,"養育院","庶務課"]
[56,"養育院","監護課"]

In [531]:
Fin=[[2,"総務局","人事課"],
     [8,"総務局","情報課"],
     [12,"財務局","主計課"],
     [26,"市民局","公園課"],
     [32,"防衛局","施設課"],
     [37,"教育局","視学課"],
     [39,"教育局","社会教育課"],
     [46,"厚生局","福利課"],
     [13,"財務局","主税課"],
     [57,"養育院","医務課"],
     [75,"土木局","治水工事課"],
     [76,"土木局","土木試験所"],
     [87,"水道局","営業課"],
     [94,"水道局","工事課"],
     [111,"電燈部","配線課"],
     [113,"電燈部","臨時電源調査課"]]

Error=[[64,"消費経済部","物価課"],
      [93,"水道局","庶務課"],
      [13,"財務局","主税課"],
      [58,"養育院","会計課"],
      [24,"市民局","町会課"],
       [36,"教育局","学務課"],
      [42,"厚生局","庶務課"],
      [71,"土木局","道路管理課"],
       [77,"土木局","土木出張所"],
       [88,"水道局","給水課"],
       [92,"水道局","下水課"],
       [95,"水道局","水源林事務所"],
       [109,"電燈部","庶務課"],
       [97,"電気局","総務課"]
      ]

In [ ]:
for Element in Fin:
    k=Element[0]
    NxDept,NxOffice=df.iloc[k+1]['Dept'],df.iloc[k+1]['Office']
    Dept,Office=df.iloc[k]['Dept'],df.iloc[k]['Office']
    
    dta[Year][Dept][Office]['EndLocation']=dta[Year][NxDept][NxOffice]['StrLocation']
    dta[Year][Dept][Office]['EndPart']=dta[Year][NxDept][NxOffice]['StrPart']
    dta[Year][Dept][Office]['EndPage']=dta[Year][NxDept][NxOffice]['StrPage']

KeyError: '気局'

In [442]:
Check(df)

[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"総務局","情報課"]
[9,"監査部","市務監察課"]
[10,"監査部","区務監察課"]
[11,"財務局","会計課"]
[12,"財務局","主計課"]
[13,"財務局","主税課"]
[14,"財務局","公債課"]
[15,"財務局","用品課"]
[16,"財務局","購買課"]
[17,"財務局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"建築部","装置課"]
[22,"市民局","庶務課"]
[23,"市民局","区政課"]
[24,"市民局","町会課"]
[25,"市民局","體力課"]
[26,"市民局","公園課"]
[27,"総動員部","第一課"]
[28,"総動員部","第二課"]
[29,"防衛局","庶務課"]
[30,"防衛局","計書課"]
[31,"防衛局","防衛課"]
[32,"防衛局","施設課"]
[33,"防衛局","防火改修課"]
[34,"教育局","庶務課"]
[35,"教育局","学校職員課"]
[36,"教育局","学務課"]
Failed at 教育局学務課
[37,"教育局","視学課"]
[38,"教育局","青年教育課"]
[39,"教育局","社会教育課"]
[40,"教育局","学校体育課"]
[41,"教育局","教育研究所"]
[42,"厚生局","庶務課"]
Failed at 厚生局庶務課
[43,"厚生局","軍事援護課"]
[44,"厚生局","児童課"]
[45,"厚生局","保護課"]
[46,"厚生局","福利課"]
[47,"厚生局","衛生課"]
[48,"厚生局","防疫課"]
[49,"清掃部","管理課"]
[50,"清掃部","作業課"]
[51,"清掃部","計書課"]
[52,"清掃部","療養所"]
[53,"清掃部","監理課"]
[54,"清掃部","醫務課"]
[55,"養育院","庶務課"]
[56,"養育院","監護課"]

In [470]:
k=0
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=90
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

消費経済部 物価課


In [377]:
k=1
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=490
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

水道局 庶務課


49

In [387]:
k=2
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=420
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

財務局 主税課


In [471]:
k=3
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=400
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

養育院 会計課


In [472]:
k=4
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=110
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

市民局 町会課


In [509]:
k=5
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=320
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

教育局 学務課


In [515]:
k=6
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=330
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

厚生局 庶務課


In [476]:
k=7
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=250
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

土木局 道路管理課


In [477]:
k=8
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=190
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

土木局 土木出張所


In [478]:
k=9
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=240
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

水道局 給水課


In [480]:
k=10
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Top'
Loc=505
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

水道局 下水課


In [519]:
k=11
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Part='Btm'
Loc=460
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part

水道局 水源林事務所


In [482]:
k=12
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)

Row,ExRow=df.iloc[n],df.iloc[n-1]
StrPage=Row['Page']
ExOffice,ExDept,EndPage=ExRow["Office"],ExRow['Dept'],ExRow['Page']

Part='Btm'
Loc=220
Show(n,Part,Loc)
cv2.destroyAllWindows()
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=Loc
dta[Year][ExDept][ExOffice]['EndLocation']=Loc
dta[Year][Dept][Office]['StrPart']=Part
dta[Year][ExDept][ExOffice]['EndPart']=Part
dta[Year][ExDept][ExOffice]['EndPage']=EndPage
dta[Year][Dept][Office]['Page_Range']=[StrPage,EndPage]

電燈部 庶務課


In [558]:
k=13
n,Office,Dept=Error[k][0],Error[k][2],Error[k][1]
print(Dept,Office)
ExRow=df.iloc[n]
ExOffice,ExDept=ExRow["Office"],ExRow['Dept']

Row=df.iloc[n]
Page=126

path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\'+Year+'\\'
stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+'Top'+".jpg",'rb')
bytes = bytearray(stream.read())
numpyarray = np.asarray(bytes, dtype=np.uint8)
img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

HH,WW=img.shape[:2]
StrLoc=370
cv2.line(img,(StrLoc,0),(StrLoc,HH),(225,0,0),2)

cv2.imshow('Projection',img)
cv2.waitKey(0)

dta[Year][Dept][Office]['StrLocation']=StrLoc
dta[Year][ExDept][ExOffice]['EndLocation']=StrLoc
dta[Year][Dept][Office]['StrPart']='Top'
dta[Year][ExDept][ExOffice]['EndPart']='Top'

電気局 総務課


In [559]:
Check(df)

[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"総務局","情報課"]
[9,"監査部","市務監察課"]
[10,"監査部","区務監察課"]
[11,"財務局","会計課"]
[12,"財務局","主計課"]
[13,"財務局","主税課"]
[14,"財務局","公債課"]
[15,"財務局","用品課"]
[16,"財務局","購買課"]
[17,"財務局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"建築部","装置課"]
[22,"市民局","庶務課"]
[23,"市民局","区政課"]
[24,"市民局","町会課"]
[25,"市民局","體力課"]
[26,"市民局","公園課"]
[27,"総動員部","第一課"]
[28,"総動員部","第二課"]
[29,"防衛局","庶務課"]
[30,"防衛局","計書課"]
[31,"防衛局","防衛課"]
[32,"防衛局","施設課"]
[33,"防衛局","防火改修課"]
[34,"教育局","庶務課"]
[35,"教育局","学校職員課"]
[36,"教育局","学務課"]
[37,"教育局","視学課"]
[38,"教育局","青年教育課"]
[39,"教育局","社会教育課"]
[40,"教育局","学校体育課"]
[41,"教育局","教育研究所"]
[42,"厚生局","庶務課"]
[43,"厚生局","軍事援護課"]
[44,"厚生局","児童課"]
[45,"厚生局","保護課"]
[46,"厚生局","福利課"]
[47,"厚生局","衛生課"]
[48,"厚生局","防疫課"]
[49,"清掃部","管理課"]
[50,"清掃部","作業課"]
[51,"清掃部","計書課"]
[52,"清掃部","療養所"]
[53,"清掃部","監理課"]
[54,"清掃部","醫務課"]
[55,"養育院","庶務課"]
[56,"養育院","監護課"]
[57,"養育院","医務課"]
[58,"養育院","会計課"]

In [ ]:
df

In [560]:
for n in range(0,len(df)):
    try:
        Row=df.iloc[n]
        ExRow=df.iloc[n-1]

        Office,Dept=Row['Office'],Row['Dept']
        ExOffice,ExDept=ExRow['Office'],ExRow['Dept']    

        StrPage=Row['Page']
        ExStrPage=ExRow['Page']

        StrLoc=dta[Year][Dept][Office]['StrLocation']

        dta[Year][ExDept][ExOffice]['EndPage']=StrPage
        dta[Year][ExDept][ExOffice]['EndLocation']=StrLoc
        
        dta[Year][ExDept][ExOffice]['Page_Range']=list(range(ExStrPage,StrPage+1))

    except:
        continue

In [561]:
Check(df)

[1,"総務局","文書課"]
[2,"総務局","人事課"]
[3,"総務局","吏務課"]
[4,"総務局","議案課"]
[5,"総務局","企画課"]
[6,"総務局","都市計画課"]
[7,"総務局","統計課"]
[8,"総務局","情報課"]
[9,"監査部","市務監察課"]
[10,"監査部","区務監察課"]
[11,"財務局","会計課"]
[12,"財務局","主計課"]
[13,"財務局","主税課"]
[14,"財務局","公債課"]
[15,"財務局","用品課"]
[16,"財務局","購買課"]
[17,"財務局","地理課"]
[18,"建築部","管理課"]
[19,"建築部","営繕課"]
[20,"建築部","学校営業課"]
[21,"建築部","装置課"]
[22,"市民局","庶務課"]
[23,"市民局","区政課"]
[24,"市民局","町会課"]
[25,"市民局","體力課"]
[26,"市民局","公園課"]
[27,"総動員部","第一課"]
[28,"総動員部","第二課"]
[29,"防衛局","庶務課"]
[30,"防衛局","計書課"]
[31,"防衛局","防衛課"]
[32,"防衛局","施設課"]
[33,"防衛局","防火改修課"]
[34,"教育局","庶務課"]
[35,"教育局","学校職員課"]
[36,"教育局","学務課"]
[37,"教育局","視学課"]
[38,"教育局","青年教育課"]
[39,"教育局","社会教育課"]
[40,"教育局","学校体育課"]
[41,"教育局","教育研究所"]
[42,"厚生局","庶務課"]
[43,"厚生局","軍事援護課"]
[44,"厚生局","児童課"]
[45,"厚生局","保護課"]
[46,"厚生局","福利課"]
[47,"厚生局","衛生課"]
[48,"厚生局","防疫課"]
[49,"清掃部","管理課"]
[50,"清掃部","作業課"]
[51,"清掃部","計書課"]
[52,"清掃部","療養所"]
[53,"清掃部","監理課"]
[54,"清掃部","醫務課"]
[55,"養育院","庶務課"]
[56,"養育院","監護課"]
[57,"養育院","医務課"]
[58,"養育院","会計課"]

Fix the department-office pair with errors

First re-run the list of failed (pairs which was rejected), with a stricter OCR.

Open FailedList_2 and see if there are any other errors left.

If there are, fix it manually by guessing the starting x axis using 'Show(n,StrLoc)'

When you find the exact starting location, update the dataframe.

dta[Year][Dept][Office]={'Starting_Page': ,
                          'Office_X1': ,
                          'Ending_Page': ,
                          'Office_X2': ,
                          'Page_Range': []}

#Step 5

Do the same thing with the department-office pair which was wrongly accepted in the test.

#Step 6

Check the entire dataframe to confirm that the lines have been drawn at the correct place.

If there are errors, add the pair into the Type1 Error list and re-run step 5.

In [562]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame_Office.json", "w") as outfile:
    outfile.write(json_object)